# Feature selection by category: Sales

Finds the strongest **sales** features (Kerendia and market NBRx, NRx, TRx, competitor share and competitor NBRx share), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with all 28 cohorts. Reads only this category's 5 source tables and fills empty values with the Combine rules.

**Note:** the writer flag is built from `TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH`, which is itself a sales feature, so it and its close relatives separate writers from non-writers by construction in the mean / median analysis.

Same method as every category notebook: the category config (cell 3) is the only difference; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4). **Part 1:** writer definition, writers / non-writers data sets, mean / median analysis (`DISCRIMINATION` = \|2 x AUC - 1\|). **Part 2:** NBA 3.1 clean-up (stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model per group on all 28 cohorts, SHAP, best features. Outputs: `feature_selection_by_category/<RUN_NAME>/sales/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "sales"
CATEGORY_LABEL = "Sales: NBRx, NRx, TRx, competitor share, competitor NBRx share"
ADD_CALENDAR   = False

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

# Rx market tables of Combine (26, 27, 28, 31) plus the competitor NBRx share table (Combine lists it under Static HCP)
SOURCE_TABLES = [
    _t("nbrx_features", "NBRX", DEV + "hcp_features_v7_cohort_aligned/nbrx_features", "delta", "cohort_id"),
    _t("nrx_features", "NRX", DEV + "hcp_features_v7_cohort_aligned/nrx_features", "delta", "cohort_id"),
    _t("trx_features", "TRX", DEV + "hcp_features_v7_cohort_aligned/trx_features", "delta", "cohort_id"),
    _t("competitor_share_features", "Competitor share", DEV + "hcp_features_v7_cohort_aligned/competitor_share_features", "delta", "cohort_id"),
    _t("competitor_nbrx_share", "Competitor NBRx share", DEV + "hcp_features/competitor_nbrx_share", "delta", "cohort_id"),
]

In [0]:
%run ./00_category_feature_selection_core